<a href="https://colab.research.google.com/github/nejumi/models-initial-course/blob/main/notebooks/02_integrations/01_integrations_overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03. Integrations: 既存コードから W&B につなぐ

## この章のゴール
Hugging Face Trainer の W&B integration を使い、自分で training loop に `run.log()` を書かなくても metric が記録されることを確認します。

## 注目する W&B コード
```python
TrainingArguments(report_to="wandb", run_name="03-hf-trainer-demo")
```

In [ ]:
# リポジトリ内ではルートを使い、Colab 単体では共通コードを取得します。
from pathlib import Path
import os
import sys
import urllib.request

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").is_file()), Path.cwd())
if not (root / "course_utils.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/nejumi/models-initial-course/main/course_utils.py",
        root / "course_utils.py",
    )
os.chdir(root)
sys.path.insert(0, str(root))

from course_utils import setup_wandb
PROJECT, ENTITY, WANDB_INIT = setup_wandb(packages=['numpy>=1.24.0', 'torch>=2.0.0', 'transformers>=4.40.0,<5', 'accelerate>=0.26.0'])

## よくある integration の形
| 形式 | 例 |
|---|---|
| `report_to="wandb"` | Hugging Face Trainer / TRL |
| `logger=WandbLogger(...)` | PyTorch Lightning |
| callback | Keras など |
| config flag | LeRobot など |

この講座で実行するのは Hugging Face Trainer です。その他は、受講後に自分の研究コードや利用中の repo で確認する発展例です。

In [ ]:
import os
import wandb
from transformers import TrainingArguments, Trainer
from course_utils import tiny_hf_trainer_objects

os.environ["WANDB_PROJECT"] = PROJECT
if ENTITY:
    os.environ["WANDB_ENTITY"] = ENTITY

model, train_dataset = tiny_hf_trainer_objects()

args = TrainingArguments(
    output_dir="outputs/03_hf_trainer",
    report_to="wandb",
    run_name="03-hf-trainer-demo",
    max_steps=5,
    per_device_train_batch_size=8,
    logging_steps=1,
    save_strategy="no",
    disable_tqdm=True,
)

trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
try:
    trainer.train()
finally:
    wandb.finish()

print("Trainer run finished")

## W&B App で確認すること
- `03-hf-trainer-demo` run が作成されている
- Trainer が `train/loss` を自動で記録している
- 自分では training loop の中に `run.log()` を書いていない